# 面试问题：Agent 长期记忆怎样合并、检索和遗忘，同时避免陈旧事实、秘密与低权威投毒？

        ## 可直接复述的回答主线

        1. 长期记忆不是把全部历史原样塞进向量库，而是区分约束、事实、决策、临时观测和敏感内容。
2. Append-only 基线会同时保留旧值和新值，检索时可能因为关键词或新近度返回错误版本。
3. Consolidation 应按 key、来源权威、有效时间和状态合并，并保留 superseded provenance。
4. Forgetting 应删除过期低价值观测和禁止持久化的秘密，但长期审批约束不能因低访问次数被遗忘。
5. 检索结果要展示值、来源、有效 turn、状态和被丢弃原因，而不是只返回相似文本。
6. 生产系统还需要租户隔离、加密、用户删除权、版本化 schema、重建索引和记忆质量评测。

        后续代码会在同一批输入上展示朴素基线、底层计算、逐步轨迹、失败复现和修正结果。

## 1. 真实案例与输入预览

案例是部署 Agent 跨会话形成的十二条脱敏记忆事件，包含区域更新、扩容上限、版本、审批要求、临时 GPU 温度、低权威检索投毒和禁止持久化的凭据占位符。五个查询用于评估当前权威事实。

In [1]:
events = [{"turn": 1, "source": "user", "kind": "constraint", "key": "region", "value": "cn-north", "importance": 0.95, "ttl": None}, {"turn": 2, "source": "user", "kind": "constraint", "key": "max_replicas", "value": 4, "importance": 0.90, "ttl": None}, {"turn": 3, "source": "tool", "kind": "fact", "key": "deployment_version", "value": "v17", "importance": 0.80, "ttl": None}, {"turn": 4, "source": "agent", "kind": "observation", "key": "gpu_temperature", "value": "84C", "importance": 0.20, "ttl": 3}, {"turn": 5, "source": "user", "kind": "constraint", "key": "approval_required", "value": True, "importance": 1.00, "ttl": None}, {"turn": 6, "source": "user", "kind": "constraint", "key": "region", "value": "cn-east", "importance": 0.98, "ttl": None}, {"turn": 7, "source": "retrieved", "kind": "fact", "key": "region", "value": "us-west", "importance": 0.70, "ttl": None}, {"turn": 8, "source": "tool", "kind": "observation", "key": "queue_depth", "value": 42, "importance": 0.35, "ttl": 2}, {"turn": 9, "source": "user", "kind": "preference", "key": "language", "value": "zh-CN", "importance": 0.75, "ttl": None}, {"turn": 10, "source": "tool", "kind": "fact", "key": "deployment_version", "value": "v18", "importance": 0.90, "ttl": None}, {"turn": 11, "source": "agent", "kind": "task", "key": "canary_rollout", "value": "done", "importance": 0.60, "ttl": 5}, {"turn": 12, "source": "tool", "kind": "secret", "key": "api_token", "value": "<redacted>", "importance": 1.00, "ttl": None}]  # 定义十二条具有来源、重要度和 TTL 的长期记忆事件。
queries = [{"id": "mem-01", "question": "当前部署区域是什么", "key": "region", "expected": "cn-east"}, {"id": "mem-02", "question": "当前部署版本是什么", "key": "deployment_version", "expected": "v18"}, {"id": "mem-03", "question": "最大副本数是多少", "key": "max_replicas", "expected": 4}, {"id": "mem-04", "question": "变更是否需要审批", "key": "approval_required", "expected": True}, {"id": "mem-05", "question": "用户语言偏好是什么", "key": "language", "expected": "zh-CN"}]  # 定义五个当前状态查询及离线期望值。
authority = {"retrieved": 1, "agent": 2, "tool": 3, "user": 4}  # 定义记忆冲突使用的来源权威等级。
current_turn = 15  # 设定评估时刻以判断临时观测是否过期。
print("教学实验输入：十二条跨会话记忆事件")  # 标记下方为脱敏离线记忆流。
print("turn source     kind         key                  value        importance ttl")  # 输出记忆字段表头。
for event in events:  # 逐条展示来源、类型、重要度和 TTL。
    print(f"{event['turn']:>4} {event['source']:<10} {event['kind']:<12} {event['key']:<20} {str(event['value']):<12} {event['importance']:>10.2f} {str(event['ttl']):>4}")  # 输出当前记忆事件。

教学实验输入：十二条跨会话记忆事件
turn source     kind         key                  value        importance ttl
   1 user       constraint   region               cn-north           0.95 None
   2 user       constraint   max_replicas         4                  0.90 None
   3 tool       fact         deployment_version   v17                0.80 None
   4 agent      observation  gpu_temperature      84C                0.20    3
   5 user       constraint   approval_required    True               1.00 None
   6 user       constraint   region               cn-east            0.98 None
   7 retrieved  fact         region               us-west            0.70 None
   8 tool       observation  queue_depth          42                 0.35    2
   9 user       preference   language             zh-CN              0.75 None
  10 tool       fact         deployment_version   v18                0.90 None
  11 agent      task         canary_rollout       done               0.60    5
  12 tool       secret       api_to

## 2. Baseline / 基线：Append-only 加关键词与新近度检索

基线永久保存十二条记录，并用字符重叠加轻微新近度选 top-1。region 的低权威 us-west 比用户更新更晚，因此会覆盖正确区域。

In [2]:
def lexical_recency_score(query, event):  # 计算问题与记忆字段的字符重叠及新近度代理。
    query_characters = set(character for character in query if "\u4e00" <= character <= "\u9fff")  # 提取问题中文字符。
    memory_text = event["key"] + str(event["value"])  # 合并记忆 key 和 value 供匹配。
    overlap = sum(character in memory_text for character in query_characters)  # 统计问题字符在记忆中的覆盖。
    key_bonus = 3 if event["key"] in {"region", "deployment_version", "max_replicas", "approval_required", "language"} else 0  # 给结构化核心字段基础权重。
    return overlap + key_bonus + 0.01 * event["turn"]  # 用新近度打破同 key 记录平票。
baseline_rows = []  # 保存五个查询的 append-only top-1。
for query in queries:  # 逐问题扫描全部历史事件。
    same_key = [event for event in events if event["key"] == query["key"]]  # 使用结构化 key 过滤候选但保留所有版本。
    top = max(same_key, key=lambda event: lexical_recency_score(query["question"], event))  # 选择当前 key 最新且相似的记录。
    baseline_rows.append({"id": query["id"], "key": query["key"], "value": top["value"], "source": top["source"], "turn": top["turn"], "correct": top["value"] == query["expected"]})  # 保存值、来源和正确性。
print("Baseline Append-only 检索")  # 标记当前输出没有 consolidation 或 forgetting。
print("查询       key                  value       source     turn  correct")  # 输出基线结果表头。
for row in baseline_rows:  # 逐查询展示命中版本和来源。
    print(f"{row['id']:<10} {row['key']:<20} {str(row['value']):<11} {row['source']:<10} {row['turn']:>4} {str(row['correct']):>8}")  # 输出当前查询 top-1。
print(f"Append-only记录数={len(events)}，包含secret={sum(event['kind'] == 'secret' for event in events)}")  # 展示无遗忘存储的规模与安全问题。

Baseline Append-only 检索
查询       key                  value       source     turn  correct
mem-01     region               us-west     retrieved     7    False
mem-02     deployment_version   v18         tool         10     True
mem-03     max_replicas         4           user          2     True
mem-04     approval_required    True        user          5     True
mem-05     language             zh-CN       user          9     True
Append-only记录数=12，包含secret=1


## 3. 底层实现：Authority-aware Consolidation 与 TTL Forgetting

secret 永不持久化；过期低重要度 observation 被遗忘；同 key 冲突先比 authority，同权威再比 turn。每个 keep、supersede、reject 和 forget 决策都进入账本。

In [3]:
def consolidate(event_stream, now):  # 把原始事件合并成当前权威长期记忆。
    memory = {}  # 保存每个 key 当前生效事件。
    ledger = []  # 保存合并、拒绝和遗忘原因。
    history = {}  # 保存每个 key 被替换版本的 provenance。
    for event in event_stream:  # 按时间顺序处理全部记忆事件。
        if event["kind"] == "secret":  # 凭据和秘密禁止进入长期存储。
            ledger.append({"turn": event["turn"], "key": event["key"], "action": "drop-secret", "reason": "禁止持久化敏感内容"})  # 记录安全丢弃决策。
            continue  # 跳过 secret。
        expired = event["ttl"] is not None and now - event["turn"] > event["ttl"]  # 判断有 TTL 事件是否已经过期。
        if expired and event["importance"] < 0.5:  # 只遗忘过期且低重要度的临时观测。
            ledger.append({"turn": event["turn"], "key": event["key"], "action": "forget-expired", "reason": "低重要度临时观测过期"})  # 记录遗忘原因。
            continue  # 不把过期观测加入长期记忆。
        previous = memory.get(event["key"])  # 读取同 key 当前权威版本。
        if previous is None:  # 首次出现的 key 直接保留。
            memory[event["key"]] = event  # 写入当前事件。
            ledger.append({"turn": event["turn"], "key": event["key"], "action": "keep", "reason": "首次权威记录"})  # 记录首次保留。
            history.setdefault(event["key"], [])  # 初始化该 key 的历史版本列表。
            continue  # 进入下一事件。
        previous_rank = authority[previous["source"]]  # 读取已有版本来源权威。
        current_rank = authority[event["source"]]  # 读取新事件来源权威。
        replace_current = current_rank > previous_rank or (current_rank == previous_rank and event["turn"] > previous["turn"])  # 更高权威优先，同权威选择更新事件。
        if replace_current:  # 当前事件可以成为新权威值。
            history[event["key"]].append(previous)  # 把旧版本移入 provenance 历史。
            memory[event["key"]] = event  # 提交新版本。
            ledger.append({"turn": event["turn"], "key": event["key"], "action": "supersede", "reason": f"{previous['source']}@{previous['turn']} 被更新"})  # 记录替换来源和 turn。
        else:  # 低权威或更旧事件不得覆盖当前值。
            history[event["key"]].append(event)  # 把被拒事件保留在审计历史。
            ledger.append({"turn": event["turn"], "key": event["key"], "action": "reject-conflict", "reason": f"保留 {previous['source']}@{previous['turn']}"})  # 记录冲突拒绝。
    return memory, history, ledger  # 返回当前记忆、版本历史和决策账本。
memory, memory_history, consolidation_ledger = consolidate(events, current_turn)  # 对十二条事件执行合并与遗忘。
print("Consolidation / Forgetting 决策账本")  # 标记下表展示每条事件去向。
print("turn key                  action             reason")  # 输出决策账本表头。
for item in consolidation_ledger:  # 逐条展示保留、替换、拒绝和遗忘。
    print(f"{item['turn']:>4} {item['key']:<20} {item['action']:<18} {item['reason']}")  # 输出当前事件的长期记忆决策。
print("当前权威记忆：", {key: (event["value"], event["source"], event["turn"]) for key, event in sorted(memory.items())})  # 展示 consolidation 后真正可检索状态。

Consolidation / Forgetting 决策账本
turn key                  action             reason
   1 region               keep               首次权威记录
   2 max_replicas         keep               首次权威记录
   3 deployment_version   keep               首次权威记录
   4 gpu_temperature      forget-expired     低重要度临时观测过期
   5 approval_required    keep               首次权威记录
   6 region               supersede          user@1 被更新
   7 region               reject-conflict    保留 user@6
   8 queue_depth          forget-expired     低重要度临时观测过期
   9 language             keep               首次权威记录
  10 deployment_version   supersede          tool@3 被更新
  11 canary_rollout       keep               首次权威记录
  12 api_token            drop-secret        禁止持久化敏感内容
当前权威记忆： {'approval_required': (True, 'user', 5), 'canary_rollout': ('done', 'agent', 11), 'deployment_version': ('v18', 'tool', 10), 'language': ('zh-CN', 'user', 9), 'max_replicas': (4, 'user', 2), 'region': ('cn-east', 'user', 6)}


## 4. 结果表与结果解读

五个查询直接按结构化 key 读取当前权威版本，并输出 provenance。合并不会抹掉旧记录，而是把它们移入历史供审计，避免同时参与在线决策。

In [4]:
consolidated_rows = []  # 保存五个查询的权威记忆结果。
for query in queries:  # 逐问题读取 consolidation 后的当前状态。
    event = memory[query["key"]]  # 按结构化 key 取得当前权威事件。
    consolidated_rows.append({"id": query["id"], "key": query["key"], "value": event["value"], "source": event["source"], "turn": event["turn"], "correct": event["value"] == query["expected"], "history_versions": len(memory_history[query["key"]])})  # 保存值、来源、版本和正确性。
baseline_accuracy = sum(row["correct"] for row in baseline_rows) / len(queries)  # 计算 append-only 当前事实准确率。
consolidated_accuracy = sum(row["correct"] for row in consolidated_rows) / len(queries)  # 计算权威合并当前事实准确率。
print("查询       Baseline值   Consolidated值  source     turn  历史版本  correct")  # 输出同查询结果对照表头。
for baseline, current in zip(baseline_rows, consolidated_rows):  # 逐查询比较追加式与权威记忆。
    print(f"{current['id']:<10} {str(baseline['value']):<12} {str(current['value']):<16} {current['source']:<10} {current['turn']:>4} {current['history_versions']:>8} {str(current['correct']):>8}")  # 输出当前字段的版本选择。
print(f"结果解读：Baseline准确率={baseline_accuracy:.1%}，Consolidated={consolidated_accuracy:.1%}；在线记忆从 {len(events)} 条事件压缩为 {len(memory)} 个当前状态。")  # 解释准确率、压缩和历史保留。

查询       Baseline值   Consolidated值  source     turn  历史版本  correct
mem-01     us-west      cn-east          user          6        2     True
mem-02     v18          v18              tool         10        1     True
mem-03     4            4                user          2        0     True
mem-04     True         True             user          5        0     True
mem-05     zh-CN        zh-CN            user          9        0     True
结果解读：Baseline准确率=80.0%，Consolidated=100.0%；在线记忆从 12 条事件压缩为 6 个当前状态。


## 5. 失败案例与修正

turn 7 的 retrieved region=us-west 比用户更新更晚，latest-wins 会接受投毒。权威规则保留 turn 6 用户明确更新的 cn-east，并把检索事件放入冲突历史。

In [5]:
latest_region = max([event for event in events if event["key"] == "region"], key=lambda event: event["turn"])  # 模拟只看最新 turn 的错误记忆。
safe_region = memory["region"]  # 读取 authority-aware consolidation 的当前区域。
region_history = [(event["value"], event["source"], event["turn"]) for event in memory_history["region"]]  # 提取被替换或拒绝的区域版本。
forgotten_keys = [item["key"] for item in consolidation_ledger if item["action"] in {"forget-expired", "drop-secret"}]  # 汇总被安全遗忘或禁止持久化的 key。
print(f"错误行为：latest-wins region={latest_region['value']} source={latest_region['source']} turn={latest_region['turn']}")  # 展示低权威新记录覆盖用户约束。
print(f"修正行为：authority-aware region={safe_region['value']} source={safe_region['source']} turn={safe_region['turn']}，history={region_history}")  # 展示当前权威值和 provenance。
print("遗忘/禁止持久化：", forgotten_keys)  # 展示临时 GPU 温度、队列和 secret 的处理结果。

错误行为：latest-wins region=us-west source=retrieved turn=7
修正行为：authority-aware region=cn-east source=user turn=6，history=[('cn-north', 'user', 1), ('us-west', 'retrieved', 7)]
遗忘/禁止持久化： ['gpu_temperature', 'queue_depth', 'api_token']


## 6. 生产边界

教学实现按结构化 key 合并，真实自然语言记忆还需实体解析、语义去重和 schema 迁移。必须提供租户隔离、加密、访问审计、用户删除权、敏感信息检测、索引重建和离线记忆回归集。

In [6]:
durable_events = len(memory)  # 统计当前在线可检索状态数量。
expired_or_secret = sum(item["action"] in {"forget-expired", "drop-secret"} for item in consolidation_ledger)  # 统计主动遗忘和安全丢弃事件。
conflict_rejections = sum(item["action"] == "reject-conflict" for item in consolidation_ledger)  # 统计低权威冲突拒绝数。
diagnostics = {"raw_events": len(events), "durable_states": durable_events, "forgotten_or_secret": expired_or_secret, "conflict_rejections": conflict_rejections, "current_accuracy": consolidated_accuracy}  # 汇总长期记忆规模、治理和质量指标。
print("生产监控快照：", diagnostics)  # 输出长期记忆系统应持续监控的低维信号。

生产监控快照： {'raw_events': 12, 'durable_states': 6, 'forgotten_or_secret': 3, 'conflict_rejections': 1, 'current_accuracy': 1.0}


## 7. 最小回归测试

只验证事件规模、当前事实、投毒阻断、秘密丢弃和结果改善。

In [7]:
assert len(events) >= 5 and len(queries) >= 5  # 保证案例包含足够多的记忆事件和评测查询。
assert safe_region["value"] == "cn-east" and safe_region["source"] == "user"  # 保证当前区域来自最新高权威用户更新。
assert latest_region["value"] != safe_region["value"]  # 保证失败案例真实复现 latest-wins 投毒。
assert "api_token" not in memory  # 保证敏感凭据占位符不会进入长期记忆。
assert "gpu_temperature" in forgotten_keys and "queue_depth" in forgotten_keys  # 保证过期低价值观测被遗忘。
assert consolidated_accuracy > baseline_accuracy  # 保证同一批当前事实查询上的结果优于 append-only。